# Esercizio 2: Confronto tra Insertion Sort e Quick Sort

**Nome:** Andrea  
**Cognome:** Petrucci  
**Matricola:** 7050922

Questo notebook analizza due algoritmi di ordinamento fondamentali: _**Insertion Sort**_ e **_Quick Sort_**. Esamineremo il loro funzionamento, complessità temporale e prestazioni pratiche attraverso esempi e confronti diretti. L'obiettivo è comprendere le differenze tra i due approcci e identificare i contesti in cui ciascun algoritmo è più efficiente.

### Prerequisiti di sistema
Per la corretta esecuzione di questo Notebook e la visualizzazione dei grafici, è necessario che nel kernel Python sia installata la libreria `matplotlib`.  
Qualora non fosse presente, è possibile installarla eseguendo il seguente comando in un terminale o in una cella temporanea:
`%pip install matplotlib`.

In [ ]:
import random
import sys
import time
import matplotlib.pyplot as plt
from typing import Callable
sys.setrecursionlimit(10000)

In [ ]:
def generate_random_array(size: int) -> list[int]:
    """Genera un array di numeri casuali con dimensione in input."""
    return [random.randint(0, 100) for _ in range(size)]

def generate_sorted_array(size: int) -> list[int]:
    """Genera un array già ordinato con dimensione in input."""
    return list(range(size))

def generate_reverse_sorted_array(size: int) -> list[int]:
    """Genera un array ordinato in modo decrescente con dimensione in input."""
    return list(range(size, 0, -1))

def generate_almost_sorted_array(size: int, swap_count: int = None) -> list[int]:
    """Genera un array quasi ordinato con dimensione in input."""
    arr = list(range(size))
    if swap_count is None:
        swap_count = random.randint(size // 100, size // 20)  # Scambia fra l'1% e il 5% degli elementi
    for _ in range(swap_count):
        i = random.randint(0, size - 1)
        j = random.randint(0, size - 1)
        arr[i], arr[j] = arr[j], arr[i]
    return arr

def generate_many_duplicates_array(size: int, value_range: int=10) -> list[int]:
    """Genera un array con molti elementi duplicati con dimensione in input."""
    return [random.randint(0, value_range) for _ in range(size)]

## Insertion Sort
Costruisce l'array ordinato un elemento alla volta, inserendo ogni nuovo elemento nella posizione corretta tra quelli già ordinati. È efficiente per array piccoli o quasi ordinati, con complessità temporale $O(n^2)$ nel caso peggiore e $O(n)$ nel caso migliore.

In [ ]:
def insertion_sort(arr: list[int]) -> None:
    """Ordina un array utilizzando l'algoritmo Insertion Sort."""
    for j in range(1, len(arr)):
        key = arr[j]
        i = j - 1
        while i >= 0 and key < arr[i]:
            arr[i + 1] = arr[i]
            i -= 1
        arr[i + 1] = key

## Quick Sort
È un algoritmo **Divide et Impera** che seleziona un elemento come pivot, partiziona l'array mettendo gli elementi minori a sinistra e i maggiori a destra, e ordina ricorsivamente i sottoarray. La versione classica possiede una complessità temporale media di $O(n \lg n)$, ma precipita a $O(n^2)$ nel caso peggiore (array già ordinati o inversamente ordinati). Per ovviare a questo limite intrinseco, in questa sede è stata implementata la versione randomizzata dell'algoritmo. L'estrazione casuale del pivot svincola le prestazioni dalla disposizione iniziale dei dati, mantenendo l'esecuzione estremamente efficiente nella pratica.

In [ ]:
def partition(A: list[int], p: int,r: int) -> int:
    i=p-1
    for j in range (p,r):
        if A[j] <= A[r]:
            i = i+1
            A[i], A[j] = A[j], A[i]
    A[i+1], A[r] = A[r], A[i+1]
    return i+1

def randomized_partition(A: list[int], p: int,r: int) -> int:
    """Sceglie un pivot casuale e lo scambia con A[r] prima della partizione."""
    i = random.randint(p, r)
    A[i], A[r] = A[r], A[i]
    return partition(A,p,r)

def quick_sort(A: list[int], p: int,r: int) -> None:
    if p<r:
        q = randomized_partition(A,p,r)
        quick_sort(A,p,q-1)
        quick_sort(A,q+1,r)

## Benchmark e Confronto
Misuriamo il tempo di esecuzione di _Insertion Sort_ e _Quick Sort_ su array di dimensioni e ordinamenti diversi e confrontiamo le prestazioni tramite tabelle e grafici. 

Per evitare la ridondanza del codice (principio DRY) e garantire la coerenza metodologica dell'esperimento, la logica di calcolo dei tempi e di generazione dei grafici è stata astratta all'interno di un'unica funzione.

In [ ]:
def run_benchmark(generatore_funzione: Callable[[int], list[int]], titolo_grafico: str) -> None:
    """
    Motore generico per eseguire il benchmark.
    - generatore_funzione: il nome della funzione che genera i dati
    - titolo_grafico: il testo da mostrare sopra il grafico
    """
    # Le dimensioni degli array da testare
    sizes = [100, 500, 1000, 2000, 5000]
    insertion_times = []
    quick_times = []

    print(f"--- BENCHMARK: {titolo_grafico} ---")
    print("Dim\tInsertion Sort\tQuick Sort")
    print("-" * 40)

    for size in sizes:
        # 1. Generiamo l'array chiamando la funzione che abbiamo passato!
        arr = generatore_funzione(size)
        
        # 2. Test Insertion Sort (su una copia)
        arr_copy1 = arr.copy()
        start = time.perf_counter()
        insertion_sort(arr_copy1)
        tempo_ins = time.perf_counter() - start
        insertion_times.append(tempo_ins)
        
        # 3. Test Quick Sort (su una copia)
        arr_copy2 = arr.copy()
        start = time.perf_counter()
        # Chiamiamo la tua funzione wrapper del quick sort
        quick_sort(arr_copy2, 0, len(arr_copy2) - 1)
        tempo_qs = time.perf_counter() - start
        quick_times.append(tempo_qs)
        
        # Stampa i tempi per questa dimensione
        print(f"{size}\t{tempo_ins:.6f}\t{tempo_qs:.6f}")
    
    # 4. Generazione del grafico per questo test
    plt.figure(figsize=(10, 6))
    plt.plot(sizes, insertion_times, marker='o', label='Insertion Sort', color='red')
    plt.plot(sizes, quick_times, marker='s', label='Quick Sort', color='blue')
    plt.xlabel("Dimensione dell'array (n)")
    plt.ylabel("Tempo di esecuzione (secondi)")
    plt.title(f"Confronto: {titolo_grafico}")
    plt.legend()
    plt.grid(True)
    plt.show()

### Benchmark 1: Array con numeri casuali

Testiamo sia _Insertion Sort_ che _Quick Sort_ su un array con numeri casuali, per confrontare le loro prestazioni.

In [ ]:
run_benchmark(generate_random_array, "Array Casuale (Valori sparsi)")

### Benchmark 2: Array Ordinato (crescente)
Testiamo sia _Insertion Sort_ che _Quick Sort_ su un array ordinato in modo crescente, per confrontare le loro prestazioni nel caso migliore per _Insertion Sort_ e quello che rappresenterebbe il caso peggiore per il _Quick Sort_ classico (con pivot fisso), evidenziando così l'efficacia della variante randomizzata qui adottata.


In [ ]:
run_benchmark(generate_sorted_array, "Array Ordinato (Valori crescenti)")

### Benchmark 3: Array Ordinato (decrescente)
Testiamo sia _Insertion Sort_ che _Quick Sort_ su un array ordinato in modo decrescente, per confrontare le loro prestazioni nel caso peggiore per _Insertion Sort_.

In [ ]:
run_benchmark(generate_reverse_sorted_array, "Array Ordinato (Valori decrescenti)")

### Benchmark 4: Array Quasi Ordinato
Testiamo sia _Insertion Sort_ che _Quick Sort_ su un array quasi ordinato, un contesto comune in molte applicazioni reali.

In [ ]:
run_benchmark(generate_almost_sorted_array, "Array Quasi Ordinato")

### Benchmark 5: Array con Molti Duplicati
Testiamo sia _Insertion Sort_ che _Quick Sort_ su un array con molti elementi duplicati, un contesto che si verifica spesso in dati reali.


In [ ]:
run_benchmark(generate_many_duplicates_array, "Array Con molti duplicati")